# 🧪 QTrader v4.7: Smart Paper Trading Engine (EV Diagnosis)

This notebook provides a safe, locally simulated trading environment using **real Coinbase market data**. It evaluates your strategy's Expected Value (EV), Win Rate, and Kelly Fraction before any real money is placed at risk.

### Rules for Going Live (`SIMULATE_MODE=False`):
1. **EV > 0**: Strategy must represent a mathematical edge.
2. **Win Rate > 52%**: Minimum threshold to reliably beat exchange fees.
3. **Slippage < 10 bps**: Avoid illiquid books.

In [7]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent))

import polars as pl
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
from datetime import datetime, timedelta

from qtrader.core.config import Config
from qtrader.core.event import OrderEvent
from qtrader.input.data.market.coinbase_market import CoinbaseMarketDataClient
from qtrader.output.execution.paper_engine import PaperTradingEngine
from qtrader.output.analytics.ev_calculator import EVCalculator

# Set dark theme for Plotly
import plotly.io as pio
pio.templates.default = "plotly_dark"

## 1. Fetch Real Market Data

In [8]:
SYMBOL = "ETH-USD"
capital = 10000.0

client = CoinbaseMarketDataClient()

# Fetch last 7 days of 1-hour candles
end_dt = datetime.now(Config.tz)
start_dt = end_dt - timedelta(days=90)

candles = client.get_candles(SYMBOL, "ONE_MINUTE", start=start_dt, end=end_dt)
print(f"Fetched {len(candles)} candles for {SYMBOL}")

book = client.get_product_book(SYMBOL, limit=1)
best_bid = book["bids"][0]["price"] if book["bids"] else 0.0
best_ask = book["asks"][0]["price"] if book["asks"] else 0.0
print(f"Current Top of Book: Bid {best_bid} | Ask {best_ask}")

Fetched 129567 candles for ETH-USD
Current Top of Book: Bid 2263.28 | Ask 2263.29


## 2. Run Paper Trading Simulation (Mean Reversion Example)

In [13]:
engine = PaperTradingEngine(starting_capital=capital)

# ── Strategy: EMA Crossover + Momentum ────────────────────────────
# Signal: Fast EMA crosses above Slow EMA + positive price momentum
# This version aims for a realistic positive edge.

import pandas as pd
df = candles.to_pandas()
df['ema_fast'] = df['close'].ewm(span=50, adjust=False).mean()
df['ema_slow'] = df['close'].ewm(span=200, adjust=False).mean()

holding = False
entry_event = None

for i in range(1, len(df)):
    row = df.iloc[i]
    prev_row = df.iloc[i-1]
    
    # 50 BTC depth is realistic for Coinbase Advanced Trade
    market_state = {
        "bid": float(row["close"]) * 0.9998,
        "ask": float(row["close"]) * 1.0002,
        "top_depth": 50.0,
        "venue": "Coinbase_ADV"
    }

    # Signal: Fast EMA crossed above Slow EMA
    signal_buy = (prev_row['ema_fast'] <= prev_row['ema_slow']) and (row['ema_fast'] > row['ema_slow'])
    signal_sell = (prev_row['ema_fast'] >= prev_row['ema_slow']) and (row['ema_fast'] < row['ema_slow'])

    if not holding and signal_buy:
        # Open LONG: 10% of capital per trade for demonstration
        qty = round((capital * 0.10) / float(row["close"]), 4)
        entry_event = OrderEvent(
            symbol=SYMBOL, order_type="MARKET", side="BUY",
            quantity=qty, price=float(row["close"])
        )
        engine.simulate_fill(entry_event, market_state)
        holding = True

    elif holding and signal_sell:
        # Close LONG
        exit_event = OrderEvent(
            symbol=SYMBOL, order_type="MARKET", side="SELL",
            quantity=entry_event.quantity, price=float(row["close"])
        )
        engine.simulate_fill(exit_event, market_state)
        holding = False
        entry_event = None

print(f"Simulation complete. Closed trades: {len(engine.closed_trades)}")


Simulation complete. Closed trades: 431


## 3. Auto-Diagnosis & Expected Value

In [16]:
calculator = EVCalculator(engine.closed_trades, fee_rate=0.0004)
diagnosis = calculator.diagnose(SYMBOL)

print("=" * 50)
print(f"🧠 STRATEGY DIAGNOSIS: {diagnosis.status} , fee_rate = {calculator.fee_rate:.2%}")
print("=" * 50)
print(f"Total Trades    : {diagnosis.total_trades}")
print(f"Win / Loss      : {diagnosis.win_count} W / {diagnosis.loss_count} L")
print(f"Win Rate        : {diagnosis.win_rate:.2%}")
print()
print(f"── EV (Expected Value) ─────────────────────────")
print(f"  EV per trade  : {diagnosis.ev_per_trade:.6f} (base currency)")
print(f"  EV %          : {diagnosis.ev_pct * 100:.4f}%  (net return per trade)")
print(f"  EV weighted   : {diagnosis.ev_weighted * 100:.4f}%  (vs capital used)")
print()
print(f"── PnL Breakdown ───────────────────────────────")
print(f"  Gross Profit  : {diagnosis.total_gross_profit:.4f}")
print(f"  Fees+Slippage : {diagnosis.total_fees_slippage:.4f}")
print(f"  Net Profit    : {diagnosis.total_net_profit:.4f}")
print()
print(f"── Quant Metrics ───────────────────────────────")
print(f"  Profit Factor : {diagnosis.profit_factor:.3f}  (target > 1.3)")
print(f"  Sharpe Ratio  : {diagnosis.sharpe_ratio:.3f}  (target > 1.5, annualised)")
print(f"  Max Drawdown  : {diagnosis.max_drawdown:.2%}")
print(f"  Kelly Fraction: {diagnosis.kelly_fraction:.4f}")
print(f"  Max Slippage  : {diagnosis.max_slippage_bps:.1f} bps")
print()
if diagnosis.warnings:
    for w in diagnosis.warnings:
        print(f"  ⚠️  {w}")


🧠 STRATEGY DIAGNOSIS: FAIL , fee_rate = 0.04%
Total Trades    : 431
Win / Loss      : 87 W / 344 L
Win Rate        : 20.19%

── EV (Expected Value) ─────────────────────────
  EV per trade  : -1.710096 (base currency)
  EV %          : -0.1710%  (net return per trade)
  EV weighted   : -0.1710%  (vs capital used)

── PnL Breakdown ───────────────────────────────
  Gross Profit  : -392.3301
  Fees+Slippage : 344.7213
  Net Profit    : -737.0514

── Quant Metrics ───────────────────────────────
  Profit Factor : 0.583  (target > 1.3)
  Sharpe Ratio  : -22.390  (target > 1.5, annualised)
  Max Drawdown  : 74.96%
  Kelly Fraction: -0.1443
  Max Slippage  : 2.2 bps

  ⚠️  Negative EV (-1.710096). Strategy loses money on average after fees+slippage.
  ⚠️  Low Profit Factor (0.58 < 1.3). Gross wins do not justify gross losses.
  ⚠️  Low Sharpe Ratio (-22.39 < 1.5). Return/risk ratio below institutional standard.
  ⚠️  Negative Kelly Fraction (-0.1443). Mathematical edge is negative — stop tra

## 4. Visualizations

In [17]:
if engine.closed_trades:
    df_trades = pd.DataFrame([vars(t) for t in engine.closed_trades])
    df_trades['cumulative_pnl'] = df_trades['pnl'].cumsum()
    
    fig = px.line(df_trades, y='cumulative_pnl', title='Paper Trading Cumulative PnL',
                  labels={'index': 'Trade #', 'cumulative_pnl': 'PnL (USD)'})
    fig.update_layout(title_font_size=20)
    fig.show()
    
    # Win/Loss scatter
    fig2 = px.scatter(df_trades, y='pnl', color=(df_trades['pnl'] > 0), 
                      title='Trade Profitability Spread',
                      labels={'color': 'Profitable', 'index': 'Trade #', 'pnl': 'PnL (USD)'})
    fig2.show()
else:
    print("No trades to visualize.")

In [12]:
# 🔍 Detailed Trade Log Audit
if engine.closed_trades:
    print(f"{'Time':<11} | {'Side':<5} | {'Entry':<10} | {'Exit':<10} | {'Net PnL':<10} | {'Slippage'}")
    print("-" * 80)
    for t in engine.closed_trades[-100:]: # Last 10 trades
        print(f"ID: {t.symbol} | {t.side:<5} | {t.entry_price:<10.2f} | {t.exit_price:<10.2f} | {t.pnl:<10.4f} | {t.slippage_bps:.1f} bps")


Time        | Side  | Entry      | Exit       | Net PnL    | Slippage
--------------------------------------------------------------------------------
ID: ETH-USD | SELL  | 1989.76    | 1984.36    | -3.5128    | 2.2 bps
ID: ETH-USD | SELL  | 1977.32    | 1972.95    | -3.0061    | 2.2 bps
ID: ETH-USD | SELL  | 1976.37    | 1972.14    | -2.9364    | 2.2 bps
ID: ETH-USD | SELL  | 1975.75    | 1974.39    | -1.4847    | 2.2 bps
ID: ETH-USD | SELL  | 1949.38    | 1943.89    | -3.6162    | 2.2 bps
ID: ETH-USD | SELL  | 1874.40    | 1895.77    | 10.5965    | 2.2 bps
ID: ETH-USD | SELL  | 1867.22    | 1859.30    | -5.0433    | 2.2 bps
ID: ETH-USD | SELL  | 1865.92    | 1842.63    | -13.2785   | 2.2 bps
ID: ETH-USD | SELL  | 1833.61    | 1821.88    | -7.1954    | 2.2 bps
ID: ETH-USD | SELL  | 1827.60    | 1822.37    | -3.6653    | 2.2 bps
ID: ETH-USD | SELL  | 1827.68    | 1823.24    | -3.2332    | 2.2 bps
ID: ETH-USD | SELL  | 1826.17    | 1823.36    | -2.3424    | 2.2 bps
ID: ETH-USD | SELL  |